# Surf Equipment Database

This notebook reads `surf_equipment.xlsx`, creates `surf_equipment.db` with 9 master tables and 2 transactional tables, validates the constraints, and generates an ERD.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

WORKBOOK = Path('surf_equipment.xlsx')
DATABASE = Path('surf_equipment.db')
ERD_DOT = Path('surf_equipment_erd.dot')
ERD_SVG = Path('surf_equipment_erd.svg')

if not WORKBOOK.exists():
    raise FileNotFoundError(WORKBOOK.resolve())

workbook = pd.ExcelFile(WORKBOOK)
sheet_names = workbook.sheet_names
if len(sheet_names) != 11:
    raise ValueError(f'Expected 11 sheets, found {len(sheet_names)}')
master_names = sheet_names[:9]
transaction_names = sheet_names[9:]
tables = {name: pd.read_excel(WORKBOOK, sheet_name=name) for name in sheet_names}
primary_keys = {name: table.columns[0] for name, table in tables.items()}
primary_key_to_table = {key: name for name, key in primary_keys.items()}

print('Master tables:', master_names)
print('Transactional tables:', transaction_names)
print('Primary keys:', primary_keys)

Master tables: ['sails', 'masts', 'extensions', 'base_pulleys', 'booms', 'rdm_sdm_shims', 'uphauls', 'harness_lines', 'boards']
Transactional tables: ['rigs', 'rig_board']
Primary keys: {'sails': 'sail_id', 'masts': 'mast_id', 'extensions': 'extension_id', 'base_pulleys': 'base_pulley_id', 'booms': 'boom_id', 'rdm_sdm_shims': 'rdm_sdm_shim_id', 'uphauls': 'uphaul_id', 'harness_lines': 'harness_line_id', 'boards': 'board_id', 'rigs': 'rig_id', 'rig_board': 'rig_board_id'}


In [2]:
def quote_identifier(identifier):
    quote = chr(34)
    return quote + str(identifier).replace(quote, quote + quote) + quote

def sqlite_type(series):
    if pd.api.types.is_integer_dtype(series):
        return 'INTEGER'
    if pd.api.types.is_numeric_dtype(series):
        return 'REAL'
    return 'TEXT'

def python_value(value):
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, 'item') else value

def create_table_sql(table_name, table):
    primary_key = table.columns[0]
    definitions = []
    for column in table.columns:
        definition = f'{quote_identifier(column)} {sqlite_type(table[column])}'
        if column == primary_key:
            definition += ' PRIMARY KEY'
        definitions.append(definition)
    if table_name in transaction_names:
        for column in table.columns[1:]:
            parent_table = primary_key_to_table.get(column)
            if parent_table and parent_table != table_name:
                parent_key = primary_keys[parent_table]
                definitions.append(f'FOREIGN KEY ({quote_identifier(column)}) REFERENCES {quote_identifier(parent_table)} ({quote_identifier(parent_key)})')
    body = ',\n    '.join(definitions)
    return f'CREATE TABLE {quote_identifier(table_name)} (\n    {body}\n);'

connection = sqlite3.connect(DATABASE)
connection.execute('PRAGMA foreign_keys = ON')
for table_name in reversed(sheet_names):
    connection.execute(f'DROP TABLE IF EXISTS {quote_identifier(table_name)}')
for table_name in sheet_names:
    connection.execute(create_table_sql(table_name, tables[table_name]))
for table_name, table in tables.items():
    columns = ', '.join(quote_identifier(column) for column in table.columns)
    placeholders = ', '.join('?' for _ in table.columns)
    rows = [[python_value(value) for value in row] for row in table.itertuples(index=False, name=None)]
    connection.executemany(f'INSERT INTO {quote_identifier(table_name)} ({columns}) VALUES ({placeholders})', rows)
connection.commit()
print(f'Created {DATABASE.resolve()}')

Created /workspaces/surf_equipment/surf_equipment.db


In [3]:
for table_name in sheet_names:
    info = connection.execute(f'PRAGMA table_info({quote_identifier(table_name)})').fetchall()
    assert [row[1] for row in info if row[5] == 1] == [primary_keys[table_name]]
    assert connection.execute(f'SELECT COUNT(*) FROM {quote_identifier(table_name)}').fetchone()[0] == len(tables[table_name])
foreign_keys = []
for table_name in transaction_names:
    foreign_keys.extend(connection.execute(f'PRAGMA foreign_key_list({quote_identifier(table_name)})').fetchall())
assert foreign_keys, 'No foreign keys were created'
assert connection.execute('PRAGMA foreign_key_check').fetchall() == []
print(f'Validated {len(sheet_names)} tables and {len(foreign_keys)} foreign-key constraints.')
for table_name in transaction_names:
    print(table_name, connection.execute(f'PRAGMA foreign_key_list({quote_identifier(table_name)})').fetchall())

Validated 11 tables and 10 foreign-key constraints.
rigs [(0, 0, 'harness_lines', 'harness_line_id', 'harness_line_id', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'uphauls', 'uphaul_id', 'uphaul_id', 'NO ACTION', 'NO ACTION', 'NONE'), (2, 0, 'rdm_sdm_shims', 'rdm_sdm_shim_id', 'rdm_sdm_shim_id', 'NO ACTION', 'NO ACTION', 'NONE'), (3, 0, 'booms', 'boom_id', 'boom_id', 'NO ACTION', 'NO ACTION', 'NONE'), (4, 0, 'extensions', 'extension_id', 'extension_id', 'NO ACTION', 'NO ACTION', 'NONE'), (5, 0, 'base_pulleys', 'base_pulley_id', 'base_pulley_id', 'NO ACTION', 'NO ACTION', 'NONE'), (6, 0, 'masts', 'mast_id', 'mast_id', 'NO ACTION', 'NO ACTION', 'NONE'), (7, 0, 'sails', 'sail_id', 'sail_id', 'NO ACTION', 'NO ACTION', 'NONE')]
rig_board [(0, 0, 'rigs', 'rig_id', 'rig_id', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'boards', 'board_id', 'board_id', 'NO ACTION', 'NO ACTION', 'NONE')]


## Entity relationship diagram

The DOT source is always written. An SVG is rendered when Graphviz is installed.

In [4]:
import shutil

def dot_id(value):
    return chr(34) + str(value).replace(chr(34), chr(92) + chr(34)) + chr(34)

def is_unique_column(table_name, column_name):
    table_info = connection.execute(f'PRAGMA table_info({quote_identifier(table_name)})').fetchall()
    if any(row[1] == column_name and row[5] == 1 for row in table_info):
        return True
    indexes = connection.execute(f'PRAGMA index_list({quote_identifier(table_name)})').fetchall()
    for index in indexes:
        index_name, is_unique = index[1], index[2]
        if is_unique:
            index_columns = connection.execute(f'PRAGMA index_info({quote_identifier(index_name)})').fetchall()
            if [row[2] for row in index_columns] == [column_name]:
                return True
    return False

dot_lines = ['digraph surf_equipment {', '  graph [rankdir=LR];', '  node [shape=plain];', '  edge [fontname="Helvetica"];']
for table_name in sheet_names:
    fields = connection.execute(f'PRAGMA table_info({quote_identifier(table_name)})').fetchall()
    rows = [f'<TR><TD BGCOLOR="#DDEBF7"><B>{table_name}</B></TD></TR>']
    for field in fields:
        marker = 'PK' if field[5] else ''
        rows.append(f'<TR><TD ALIGN="LEFT">{marker} {field[1]} : {field[2]}</TD></TR>')
    label = '<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="5">' + ''.join(rows) + '</TABLE>>'
    dot_lines.append(f'  {dot_id(table_name)} [label={label}];')
for table_name in transaction_names:
    for foreign_key in connection.execute(f'PRAGMA foreign_key_list({quote_identifier(table_name)})'):
        parent_table, parent_column, child_column = foreign_key[2], foreign_key[4], foreign_key[3]
        child_cardinality = '1' if is_unique_column(table_name, child_column) else '∞'
        dot_lines.append(
            f'  {dot_id(parent_table)} -> {dot_id(table_name)} '
            f'[label="{child_column} -> {parent_column}", taillabel="1", headlabel="{child_cardinality}"];'
        )
dot_source = '\n'.join(dot_lines + ['}']) + '\n'
ERD_DOT.write_text(dot_source, encoding='utf-8')
if shutil.which('dot'):
    try:
        import graphviz
        graphviz.Source(dot_source).render(filename=str(ERD_SVG.with_suffix('')), format='svg', cleanup=True)
        print(f'Created {ERD_SVG.resolve()}')
    except Exception as error:
        print(f'SVG rendering failed: {error}')
else:
    print('Graphviz dot executable not found; use the DOT ERD source or install Graphviz to render SVG.')
print(f'Created {ERD_DOT.resolve()}')

Created /workspaces/surf_equipment/surf_equipment_erd.svg
Created /workspaces/surf_equipment/surf_equipment_erd.dot


In [5]:
connection.close()
print('Database connection closed.')

Database connection closed.
